In [7]:
import ee
import geemap
import os


In [ ]:

# 1. GEE 인증 및 초기화
ee.Initialize(project='aerobic-gift-464705-u5')
# 2. 한반도 관심 지역 설정 (예: 전라남도 나주/광주 인근)
roi = ee.Geometry.Rectangle([126.5, 34.8, 127.2, 35.3])

# 3. Sentinel-2 및 s2cloudless (Fmask) 컬렉션 결합
def get_s2_with_cloud_mask(roi, start_date, end_date):
    # S2 SR 데이터
    s2_sr = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
             .filterBounds(roi)
             .filterDate(start_date, end_date))
    
    # s2cloudless 데이터 (구름 확률 맵)
    s2_cloud = (ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
                .filterBounds(roi)
                .filterDate(start_date, end_date))
    
    # 두 컬렉션을 ID 기준으로 결합
    return ee.ImageCollection(ee.Join.saveFirst('cloud_mask').apply(
        primary=s2_sr,
        secondary=s2_cloud,
        condition=ee.Filter.equals(leftField='system:index', rightField='system:index')
    ))

# 4. Sentinel-1 (VV + VH) 매칭 함수
def add_s1_dual_pol(s2_img):
    s2_img = ee.Image(s2_img)
    date = s2_img.date()
    
    # 1. Sentinel-1 컬렉션 필터링
    s1_col_filtered = (ee.ImageCollection("COPERNICUS/S1_GRD")
                .filterBounds(roi)
                .filter(ee.Filter.eq('instrumentMode', 'IW'))
                .filterDate(date.advance(-3, 'day'), date.advance(3, 'day')))
    
    # 2. 날짜 차이 계산 (단위를 'day'로 변경하여 호환성 확보)
    s1_match = ee.Image(s1_col_filtered
                .map(lambda img: img.set('dateDist', img.date().difference(date, 'day').abs()))
                .sort('dateDist') 
                .first())
    
    # 3. Fmask 대용 구름 확률 밴드 추가
    # get('cloud_mask')가 이미지를 반환하지 않을 경우를 대비해 ee.Image()로 한 번 더 감쌉니다.
    cloud_prob = ee.Image(s2_img.get('cloud_mask')).select('probability').rename('Fmask_prob')
    
    # 4. 최종 밴드 통합 (7개 채널)
    # S2(B4, B3, B2, B8) + S1(VV, VH) + Fmask(prob)
    return s2_img.select(['B4', 'B3', 'B2', 'B8']).addBands(s1_match.select(['VV', 'VH'])).addBands(cloud_prob)
    
# 실행 및 샘플 확인
# 5. 실행 및 샘플 확인
start_date = '2023-08-01'
end_date = '2023-08-31'
final_collection = get_s2_with_cloud_mask(roi, start_date, end_date).map(add_s1_dual_pol)

# 첫 번째 이미지 샘플 정보 확인
sample_img = final_collection.first()
print("포함된 밴드 목록:", sample_img.bandNames().getInfo()) 
# 출력 결과: ['B4', 'B3', 'B2', 'B8', 'VV', 'VH', 'Fmask_prob']

In [9]:
output_path = os.path.expanduser('~/data_2/SARtoRGB/Korea')

os.makedirs(output_path, exist_ok=True)
# 2. 파일명 설정
filename = os.path.join(output_path, 'korea_7band_patch_sample.tif')

# 3. 로컬로 직접 다운로드 (scale=10은 10m 해상도)
# sample_img는 앞서 만든 ['B4', 'B3', 'B2', 'B8', 'VV', 'VH', 'Fmask_prob'] 포함 이미지
geemap.ee_export_image(
    sample_img, 
    filename=filename, 
    scale=10, 
    region=roi, 
    file_per_band=False # 모든 밴드를 하나의 파일에 합쳐서 저장
)

print(f"데이터가 다음 경로에 성공적으로 저장되었습니다: {filename}")

Generating URL ...
An error occurred while downloading.
Total request size (1183181440 bytes) must be less than or equal to 50331648 bytes.
데이터가 다음 경로에 성공적으로 저장되었습니다: /Users/choeuiyeong/data_2/SARtoRGB/Korea/korea_7band_patch_sample.tif


In [ ]:
geemap.ee_export_image_collection(
    final_collection, 
    out_dir=output_path, 
    format='ZIPPED_GEO_TIFF', 
    scale=10, 
    region=roi,
    file_per_band=False # 7개 밴드를 하나의 파일에 묶어서 저장
)

print(f"패치 추출 작업이 완료되었습니다. 경로를 확인하세요: {output_path}")

Total number of images: 20

Exporting 1/20: /Users/choeuiyeong/data_2/SARtoRGB/Korea/20230805T021539_20230805T022738_T52SBD.tif
Generating URL ...
An error occurred while downloading.
Collection.toList: Error in map(ID=20230820T021541_20230820T022103_T52SBD):
Image.select: Parameter 'input' is required and may not be null.


Exporting 2/20: /Users/choeuiyeong/data_2/SARtoRGB/Korea/20230805T021539_20230805T022738_T52SBE.tif
Generating URL ...
An error occurred while downloading.
Collection.toList: Error in map(ID=20230820T021541_20230820T022103_T52SBE):
Image.select: Parameter 'input' is required and may not be null.


Exporting 3/20: /Users/choeuiyeong/data_2/SARtoRGB/Korea/20230805T021539_20230805T022738_T52SCD.tif
Generating URL ...
An error occurred while downloading.
Collection.toList: Error in map(ID=20230820T021541_20230820T022103_T52SBE):
Image.select: Parameter 'input' is required and may not be null.


Exporting 4/20: /Users/choeuiyeong/data_2/SARtoRGB/Korea/20230805T021539_20

NameError: name 'output_dir' is not defined